# 2 — Train the captioner on Kaggle

Drives `ml/train.py` end to end on a Kaggle GPU runtime:

1. install requirements and clone this repo
2. locate the COCO images and the Karpathy split
3. build `train.json` / `val.json` / `test.json` / `vocab.json` if missing
4. train, writing to `/kaggle/working/run1`
5. resume from `last_state.pt` after an interruption
6. plot `metrics.csv`

Everything is driven from the **Config** cell below, so nothing needs editing
further down. Attach a GPU runtime first: Settings → Accelerator → GPU (T4).

## 1. Install

In [ ]:
# Kaggle ships several packages that collide with ours; drop them first.
# Remove this block if you are not on Kaggle.
!pip uninstall -y -q nltk spacy 2>/dev/null | tail -2 || true

In [ ]:
!git clone https://github.com/Arvind-kumar2006/Attention_base_image_captioning.git /kaggle/working/repo \
  || echo "already cloned — skipping"

In [ ]:
# The notebook can be opened from anywhere, so make sure the code is on disk
# before anything tries to import it.
import os, subprocess

REPO = "/kaggle/working/repo"

if not os.path.isdir(os.path.join(REPO, "ml")):
    print("ml/ not present in the clone, fetching the code ...")
    subprocess.run(["git", "-C", REPO, "fetch", "--all"], check=False)
    subprocess.run(
        ["git", "-C", REPO, "checkout", "FETCH_HEAD", "--", "ml", "backend", ".gitignore"],
        check=False,
    )

if not os.path.isdir(os.path.join(REPO, "ml")):
    raise SystemExit(
        f"no ml/ directory under {REPO}. Check that the clone URL in the cell above "
        "points at the branch holding the code."
    )

print("repo contents:", sorted(os.listdir(REPO))[:12])

## 2. Install the ML package

Editable install so `ml/` and the `captioning` package the backend imports are
the same code.

In [ ]:
%cd /kaggle/working/repo
!pip install -q -r ml/requirements.txt
!pip install -q -e ml/
print("installed")

## 3. Config

Change `REPO`, `WORK` and the dataset settings here; every later cell reads
from this one.

In [ ]:
import os, sys, json, csv, subprocess, time, zipfile, urllib.request
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from PIL import Image

REPO      = "/kaggle/working/repo"
REPO_DIR  = Path(REPO)       # same path as a Path, for joins
WORK      = Path("/kaggle/working")
RUN_DIR   = WORK / "run1"          # checkpoints + metrics.csv land here
DATA_DIR  = WORK / "processed"     # train.json / val.json / test.json / vocab.json
KARPATHY  = WORK / "dataset_coco.json"

# Candidate folders that may hold train2014/ and val2014/. Kaggle datasets mount
# under /kaggle/input, so add your own there if you use one.
COCO_ROOT_CANDIDATES = [
    "/kaggle/input/coco2014",
    "/kaggle/input/coco-2014",
    "/kaggle/working/data/coco",
    "/kaggle/working/coco",
]
COCO_ROOT_OVERRIDE = None          # set to a str/Path to skip the search

# Training settings passed to ml/train.py.
TRAIN = {
    "epochs": 5,
    "batch_size": 32,
    "lr": 4e-4,
    "workers": 4,
    "amp": True,
    "subset": None,                # e.g. 20000 for a quick first run
    "finetune_after": None,        # e.g. 3 to unfreeze layer3/layer4 at epoch 3
    "patience": 3,
}
FINETUNE_LR = 1e-4                 # encoder lr once fine-tuning starts

# make the repo importable from anywhere in the notebook
if REPO not in sys.path:
    sys.path.insert(0, REPO)

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("device     :", "cuda" if torch.cuda.is_available() else "cpu")
print("REPO       :", REPO_DIR)
print("RUN_DIR    :", RUN_DIR)
print("DATA_DIR   :", DATA_DIR)
print("train cfg  :", TRAIN)

## 4. Find the COCO root and the Karpathy split

`data_prep.py` skips images it cannot find, so a wrong root silently shrinks the training set. Check it before building.

In [ ]:
import os

def has_jpgs(folder):
    """True when the directory exists and holds at least one image."""
    try:
        with os.scandir(folder) as entries:
            for entry in entries:
                if entry.name.lower().endswith((".jpg", ".jpeg")):
                    return True
    except FileNotFoundError:
        pass
    return False


def find_coco_root(bases, max_depth=6):
    """Find the folder containing train2014/ and/or val2014/ with images in it."""
    for base in bases:
        if not os.path.isdir(base):
            continue
        base_depth = base.rstrip("/").count("/")
        for dirpath, dirnames, _files in os.walk(base):
            if dirpath.rstrip("/").count("/") - base_depth >= max_depth:
                dirnames[:] = []
                continue
            for name in ("val2014", "train2014"):
                if name in dirnames and has_jpgs(Path(dirpath) / name):
                    return Path(dirpath)
    return None


COCO_ROOT = Path(COCO_ROOT_OVERRIDE) if COCO_ROOT_OVERRIDE else find_coco_root(COCO_ROOT_CANDIDATES)
assert COCO_ROOT is not None, (
    "COCO images not found. Attach the dataset as a Kaggle input dataset, or set "
    "COCO_ROOT_OVERRIDE to the folder holding train2014/ and val2014/."
)
print("COCO root:", COCO_ROOT)
for name in ("train2014", "val2014"):
    folder = COCO_ROOT / name
    count = len(list(folder.glob("*.jpg"))) if folder.is_dir() else 0
    print(f"  {name}: {count:,} images")

In [ ]:
# The Karpathy split file (~450 MB of annotations). Fetched once and cached.
import urllib.request, zipfile

KARPATHY_URL = "https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip"

if not KARPATHY.exists():
    print("downloading the Karpathy split, this takes a few minutes ...")
    zip_path = WORK / "caption_datasets.zip"
    urllib.request.urlretrieve(KARPATHY_URL, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        member = next(n for n in archive.namelist() if n.endswith("dataset_coco.json"))
        with archive.open(member) as src, open(KARPATHY, "wb") as dst:
            dst.write(src.read())
    zip_path.unlink(missing_ok=True)

print(f"split file: {KARPATHY} ({KARPATHY.stat().st_size / 1e6:.0f} MB)")

## 5. Build the splits

Skipped when the JSON files already exist, so re-running the notebook after a
disconnect does not rebuild 20 GB of metadata. Delete `DATA_DIR` to force it.

In [ ]:
import subprocess, sys, time

NEEDED = ("train.json", "val.json", "test.json", "vocab.json")
missing = [name for name in NEEDED if not (DATA_DIR / name).is_file()]

if not missing:
    print("splits already built, skipping data_prep:", NEEDED)
else:
    print("missing:", missing)
    started = time.time()
    command = [
        sys.executable, "-m", "ml.data_prep",
        "--karpathy-json", str(KARPATHY),
        "--coco-root", str(COCO_ROOT),
        "--out-dir", str(DATA_DIR),
    ]
    print(" ".join(command))
    result = subprocess.run(command, cwd=REPO)
    if result.returncode != 0:
        raise SystemExit(f"data_prep failed with code {result.returncode}")
    print(f"data_prep finished in {(time.time() - started) / 60:.1f} min")

In [ ]:
for name in NEEDED:
    path = DATA_DIR / name
    if path.is_file():
        payload = json.loads(path.read_text())
        size = len(payload)
        extra = f", vocab {size:,} words" if name == "vocab.json" else f", {sum(len(i['caps']) for i in payload):,} captions"
        print(f"  {name:>12}: {size:,}{extra}")

## 6. Sanity check

`ml.train --overfit` on a handful of images is the fastest way to prove the whole
path works: real encoder, real decoder, real optimiser. It runs on CPU or GPU in
a couple of minutes. Delete the folder afterwards so it cannot be mistaken for a
real run.

In [ ]:
# comment this cell out once the real run below works
SANITY_DIR = WORK / "sanity"

command = [
    sys.executable, "-m", "ml.train",
    "--coco-root", str(COCO_ROOT),
    "--data-dir", str(DATA_DIR),
    "--out-dir", str(SANITY_DIR),
    "--overfit", "8",
    "--epochs", "3",
    "--batch-size", "8",
    "--workers", "0",
]
print(" ".join(command))
result = subprocess.run(command, cwd=REPO)

## 7. Train

One config cell, one call. `--resume` reuses the same function with the state
file, so there is no second code path to keep in sync.

Each epoch writes `last.pt`, `best.pt` when BLEU-4 improves, `metrics.csv`, and
`last_state.pt` for resuming.

In [ ]:
def run_train(out_dir, epochs=None, resume=False, extra=None, **overrides):
    """Call ml/train.py with the config from cell 3.

    Args:
        out_dir: where checkpoints and metrics.csv are written.
        epochs: overrides TRAIN["epochs"].
        resume: continue from ``<out_dir>/last_state.pt`` if it exists.
        extra: extra CLI flags, e.g. ["--overfit", "8"].
        **overrides: replace individual TRAIN entries, e.g. batch_size=8.

    Returns:
        The subprocess return code.
    """
    settings = {**TRAIN, **overrides}
    if epochs is not None:
        settings["epochs"] = epochs

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    state_file = out_dir / "last_state.pt"

    command = [
        sys.executable, "-m", "ml.train",
        "--coco-root", str(COCO_ROOT),
        "--data-dir", str(DATA_DIR),
        "--out-dir", str(out_dir),
        "--epochs", str(settings["epochs"]),
        "--batch-size", str(settings["batch_size"]),
        "--lr", str(settings["lr"]),
        "--workers", str(settings["workers"]),
        "--patience", str(settings["patience"]),
    ]
    if settings.get("amp"):
        command.append("--amp")
    if settings.get("subset"):
        command += ["--subset", str(settings["subset"])]
    if settings.get("finetune_after"):
        command += ["--finetune-after", str(settings["finetune_after"])]
    if resume and state_file.is_file():
        command += ["--resume", str(state_file)]
        print(f"resuming from {state_file}")
    elif resume:
        print(f"no state file at {state_file}, starting fresh")
    if extra:
        command += [str(flag) for flag in extra]

    print(" ".join(command), "\n")
    return subprocess.run(command, cwd=REPO).returncode

In [ ]:
returncode = run_train(RUN_DIR)
if returncode != 0:
    raise SystemExit(f"training failed with code {returncode}")

print("artifacts in", RUN_DIR)
for path in sorted(RUN_DIR.iterdir()):
    print(f"  {path.name:>18}  {path.stat().st_size / 1e6:8.2f} MB")

## 8. Resume

Kaggle kills long runs at 12 hours. `--resume` restores the model weights,
optimiser state, epoch and best BLEU-4 from `last_state.pt`, so training
continues where it stopped instead of restarting.

Run this cell as many times as needed; each invocation picks up from the last
completed epoch.

`last.pt` and `best.pt` follow the checkpoint contract and stay resumable on
their own — only the optimiser state lives in `last_state.pt`.

In [ ]:
# picks up from the last completed epoch, or starts fresh if there is none
returncode = run_train(RUN_DIR, resume=True)
if returncode != 0:
    raise SystemExit(f"resumed training failed with code {returncode}")

In [ ]:
# If the session was restarted, last_state.pt lives in the run folder, not in
# memory. Rehydrating the handles keeps the next cell working either way.
RESUME_STATE = RUN_DIR / "last_state.pt"
if RESUME_STATE.is_file():
    payload = torch.load(RESUME_STATE, map_location="cpu", weights_only=False)
    print(f"resumable from epoch {payload['epoch']}, best BLEU-4 {payload['best_bleu4']:.4f}")
    print(f"fine-tuning active: {payload.get('finetune_applied', False)}")
else:
    print("no state file yet, finish cell 7 first")

## 9. Plot metrics

Loss and the BLEU-4 proxy per epoch. The proxy is nltk corpus BLEU over greedy
decoding on a fixed 1000-image validation sample — good for spotting a plateau,
not a number to report. `ml/evaluate.py` produces the real figures on the test
split.

In [ ]:
import csv
import matplotlib.pyplot as plt

METRICS = RUN_DIR / "metrics.csv"
if not METRICS.is_file():
    raise SystemExit(f"{METRICS} not found, run the training cell first")

with METRICS.open() as handle:
    rows = list(csv.DictReader(handle))

epochs = [int(row["epoch"]) for row in rows]
train_loss = [float(row["train_loss"]) for row in rows]
val_loss = [float(row["val_loss"]) for row in rows]
val_bleu4 = [float(row["val_bleu4"]) for row in rows]
minutes = [float(row["minutes"]) for row in rows]
memory = [float(row["gpu_memory_mb"]) for row in rows]

print(f"{len(rows)} epochs | total {sum(minutes):.1f} min | peak GPU {max(memory):.0f} MiB")
print(f"train loss {train_loss[0]:.4f} -> {train_loss[-1]:.4f}")
print(f"val   loss {val_loss[0]:.4f} -> {val_loss[-1]:.4f}")
print(f"val BLEU-4 best {max(val_bleu4):.4f} (epoch {epochs[val_bleu4.index(max(val_bleu4))]})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs, train_loss, marker="o", label="train")
axes[0].plot(epochs, val_loss, marker="o", label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy")
axes[0].set_title("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(epochs, val_bleu4, marker="o", color="tab:green", label="val BLEU-4 (nltk proxy)")
axes[1].axhline(max(val_bleu4), ls="--", lw=1, color="tab:green", alpha=0.5)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("BLEU-4")
axes[1].set_title("Validation BLEU-4 proxy")
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.suptitle(f"run1 — {len(rows)} epochs, {sum(minutes):.0f} min total, peak {max(memory):.0f} MiB")
fig.tight_layout()
plt.savefig(RUN_DIR / "metrics.png", dpi=120, bbox_inches="tight")
plt.show()

## 10. Greedy captions

A quick look at what the model produces. Untrained or barely trained output is
gibberish, which is normal — it should start forming words within an epoch.

In [ ]:
# `captioning` lives in ml/, which the editable install already put on the path;
# this line keeps the cell working if the install was skipped.
sys.path.insert(0, str(REPO_DIR / "ml"))

from captioning.generator import CaptionGenerator
from captioning.decoding import BeamSearchDecodingStrategy

CHECKPOINT = RUN_DIR / "best.pt"
if not CHECKPOINT.is_file():
    CHECKPOINT = RUN_DIR / "last.pt"
if not CHECKPOINT.is_file():
    raise SystemExit("no checkpoint yet, run the training cell first")

device = "cuda" if torch.cuda.is_available() else "cpu"
generator = CaptionGenerator(CHECKPOINT, device=device)
generator.strategy = BeamSearchDecodingStrategy(generator.model.decoder, beam_size=3)
print(f"{CHECKPOINT.name} on {device} | beam 3 | vocab {len(generator.vocab):,} words")

In [ ]:
items = json.loads((DATA_DIR / "val.json").read_text())
for item in items[:5]:
    image = Image.open(COCO_ROOT / item["path"]).convert("RGB")
    result = generator.generate(image)
    print(f"image    : {item['path']}")
    print(f"generated: {result.caption}")
    for tokens in item["caps"][:2]:
        print(f"reference: {' '.join(tokens)}")
    print(f"attention: {result.attention.shape}, rows sum to "
          f"{result.attention.sum(axis=(1, 2)).mean():.4f}, {result.time_ms:.0f} ms\n")

## 11. Score the test split

Real BLEU-1..4, METEOR and CIDEr with pycocoevalcap. Needs the images and the
Karpathy annotations already downloaded above.

`PTBTokenizer` needs Java, which Kaggle provides; without it the script falls
back to whitespace tokenisation and says so.

In [ ]:
RESULTS = RUN_DIR / "results.json"
if not (DATA_DIR / "test.json").is_file() or not json.loads((DATA_DIR / "test.json").read_text()):
    raise SystemExit(
        "the test split is empty. The Karpathy annotations need images that are "
        "present under COCO_ROOT, and val/restval are merged into train — check "
        "the cell 5 output for skipped entries."
    )

command = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(RUN_DIR / "best.pt"),
    "--data-dir", str(DATA_DIR),
    "--coco-root", str(COCO_ROOT),
    "--split", "test",
    "--beam-size", "3",
    "--out", str(RESULTS),
]
print(" ".join(command), "\n")
returncode = subprocess.run(command, cwd=REPO).returncode
if returncode != 0 or not RESULTS.is_file():
    raise SystemExit(f"evaluate.py failed with code {returncode}; see the output above")
print("wrote", RESULTS)

In [ ]:
if not RESULTS.is_file():
    raise SystemExit(f"{RESULTS} not found, run the scoring cell first")

results = json.loads(RESULTS.read_text())
print("tokenizer:", results["tokenizer"])
print(f"images   : {results['dataset']['num_images']:,}")
for name, value in results["scores"].items():
    print(f"  {name:>8}: {value:.4f}")
print(f"\n{len(results['captions']):,} captions recorded in {RESULTS.name}")

In [ ]:
# Download the artifacts before the session ends, or the run is lost with it.
artifact = RUN_DIR / "artifacts.zip"
patterns = ["*.pt", "*.json", "*.csv", "*.png"]

files = [p for pattern in patterns for p in sorted(RUN_DIR.glob(pattern))]
subprocess.run(["zip", "-j", "-q", str(artifact), *[str(p) for p in files]], check=False)
print(f"{len(files)} files, {artifact.stat().st_size / 1e6:.1f} MB")

from IPython.display import FileLink

FileLink(str(artifact))